**GroupDNA - WhatsApp Group Chat Analyzer**

**Project Name:** GroupDNA - Your WhatsApp Group, Decoded

**Student Name:** G. Mohit

**Roll Number:** Nill

**Batch:** Data Science June 2026

**Date:** June 27, 2026


**Project Overview**

This project analyzes a WhatsApp group chat export to generate a comprehensive personality and activity report. It identifies patterns in messaging behavior, assigns personality archetypes to participants and visualizes group dynamics through text-based analytics.

**Dataset:** hostel_bois.txt (synthetic WhatsApp chat export)

**Participants:** 6 (Priya, Rahul, Aman, Vikas, Karan, Neha,)

**Date Range:** 01 April 2024 to 30 May 2024 (60 days)

**Total Messages:** 3,174

**Feature 1: Chat Parser**

This feature reads the WhatsApp chat export file line by line, extracts timestamp, sender and message text and also handles edge cases like system messages, media-omitted and deleted messages.

In [ ]:
# Feature 1: Chat Parser
# AI-assisted: Asked AI how to import libraries for the date and time and parse the chat.
from datetime import datetime # Imported the Date and Time from the datetime library.
import numpy as np # Imported the numpy library as np.
def parse_chat(file_path): # User defined function for the the chat parser.
    messages = [] # To store the messages.
    stats = {  # To store the edge cases for handling it.
        'total_lines': 0,
        'real_messages': 0,
        'system_messages': 0,
        'media_omitted': 0,
        'deleted_messages': 0
    }
    with open(file_path, 'r', encoding='utf-8') as f: # Opens the file for reading it.
        content = f.read() # Reads the content of the file line by line.
    lines = content.split('\n') # Splits the lines of the contents in the file after reading it.
    stats['total_lines'] = len(lines) # Stores the length/total count of the lines.
    for line in lines:
        if not line.strip(): # Skip the empty lines.
            continue
        if len(line) < 8 or line[2] != '/' or line[5] != '/': # Check if the line starts with date pattern in the format (DD/MM/YY).
            continue # This is a continuation line for multi-line message and since in our dataset all messages are single-line so we skip.
        try: # Split timestamp from the rest of the line
            timestamp_part, rest = line.split(' - ', 1)
        except ValueError: # In case try doesnt the except ValueError runs.
            continue
        if ':' not in rest: # Check for the system messages which should have no colon after timestamp.
            stats['system_messages'] = stats['system_messages'] + 1 # increment the stats of the system messages by 1
            continue
        try: # Split the sender from the message.
            sender, message = rest.split(': ', 1)
        except ValueError:
            continue
        if '<Media omitted>' in message: # Check for the media-omitted in the message.
            stats['media_omitted'] = stats['media_omitted'] + 1 # increment the stats of the media ommited by 1.
            continue
        if 'This message was deleted' in message: # Check for the deleted messages in message.
            stats['deleted_messages'] = stats['deleted_messages'] + 1 # increment the stats of the deleted messages by 1.
            continue
        messages.append({ # Valid message - add to list.
            'timestamp': timestamp_part,
            'sender': sender,
            'message': message
        })
        stats['real_messages'] = stats['real_messages'] + 1 # increment the stats of the real messages by 1.
    return messages, stats
messages, parse_stats = parse_chat('hostel_bois.txt') # Call the function parse_chat to parse the chat file.
print(f"Successfully parsed {parse_stats['real_messages']} messages from {len(set(m['sender'] for m in messages))} participants over 60 days")
print(f"Skipped: {parse_stats['system_messages']} system messages, {parse_stats['media_omitted']} media-omitted, {parse_stats['deleted_messages']} deleted messages")

Successfully parsed 3127 messages from 6 participants over 60 days
Skipped: 4 system messages, 32 media-omitted, 15 deleted messages


**Feature 2: Group Overview**

Computes and displays the headline statistics about the group including total messages, date range, participants and per person message counts.

In [ ]:
# Feature 2: Group Overview
def compute_group_overview(messages): # User defined function to compute the group overview.
    if not messages:
        return {}
    participants = set(m['sender'] for m in messages) # Receives the unique participants.
    message_counts = {} # Counts the messages per person.
    for msg in messages:
        sender = msg['sender']
        message_counts[sender] = message_counts.get(sender, 0) + 1
    sorted_counts = sorted(message_counts.items(), key=lambda x: x[1], reverse=True) # Sorts by message count in descending order.
    timestamps = [datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M') for m in messages] # Receives the date range.
    first_date = min(timestamps) # Stores the starting date.
    last_date = max(timestamps) # Stores the ending date.
    total_days = (last_date - first_date).days + 1 # Calculate the total number of days.
    return {
        'group_name': 'Hostel Bois 4ever',
        'first_date': first_date.strftime('%d %B %Y'),
        'last_date': last_date.strftime('%d %B %Y'),
        'total_days': total_days,
        'total_messages': len(messages),
        'participants': len(participants),
        'message_counts': sorted_counts
    }
overview = compute_group_overview(messages) # Computes the Group Overview.
print("=" * 60) # Prints the formatted overview.
print("  GROUP OVERVIEW")
print("=" * 60)
print(f"  Group           : {overview['group_name']}")
print(f"  Period          : {overview['first_date']} to {overview['last_date']} ({overview['total_days']} days)")
print(f"  Total messages  : {overview['total_messages']:,}")
print(f"  Participants    : {overview['participants']}")
print()
print("  MESSAGES PER PERSON")
total_msgs = overview['total_messages']
for sender, count in overview['message_counts']:
    percentage = (count / total_msgs) * 100
    print(f"    {sender:<10} : {count:>4}  ({percentage:>5.1f}%)")

  GROUP OVERVIEW
  Group           : Hostel Bois 4ever
  Period          : 01 April 2024 to 30 May 2024 (60 days)
  Total messages  : 3,127
  Participants    : 6

  MESSAGES PER PERSON
    Rahul      :  940  ( 30.1%)
    Priya      :  712  ( 22.8%)
    Neha       :  624  ( 20.0%)
    Aman       :  484  ( 15.5%)
    Karan      :  345  ( 11.0%)
    Vikas      :   22  (  0.7%)


**Feature 3 : Most Active Day And Hour**

Computes the most active or busiest day and hour of the whatsapp group.

In [ ]:
# Feature 3 : Most active day and hour
def find_most_active_periods(messages):
    if not messages:
        return {}
    day_counts = {} # Count of the messages per day
    hour_counts = {}
    unique_dates = set()  # Get all the unique dates to calculate the total days
    for msg in messages:
        timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        date = timestamp.date()
        hour = timestamp.hour
        unique_dates.add(date)
        day_str = date.strftime('%d %B %Y') # Count per day
        day_counts[day_str] = day_counts.get(day_str, 0) + 1
        hour_counts[hour] = hour_counts.get(hour, 0) + 1 # Count per hour
    most_active_day = max(day_counts.items(), key=lambda x: x[1]) # Find the most active day
    most_active_hour = max(hour_counts.items(), key=lambda x: x[1])  # Find the most active hour
    total_days = len(unique_dates)  # Calculate the total days for average calculation
    avg_messages_per_day = most_active_hour[1] / total_days if total_days > 0 else 0
    return {
        'most_active_day': {
            'date': most_active_day[0],
            'count': most_active_hour[1]
        },
        'most_active_hour': {
            'hour': most_active_hour[0],
            'count': most_active_hour[1],
            'avg_per_day': avg_messages_per_day
        },
        'day_counts': day_counts,
        'hour_counts': hour_counts
    }
activity_periods = find_most_active_periods(messages) # Find the most active periods
print(f"Busiest day  : {activity_periods['most_active_day']['date']} ({activity_periods['most_active_day']['count']} messages)") # Print formatted output matching the requested format
hour = activity_periods['most_active_hour']['hour']
next_hour = (hour + 1) % 24
avg = activity_periods['most_active_hour']['avg_per_day']
print(f"Busiest hour : {hour:02d}:00 - {next_hour:02d}:00 (avg {avg:.1f} messages per day)")

Busiest day  : 04 May 2024 (244 messages)
Busiest hour : 18:00 - 19:00 (avg 4.1 messages per day)


**Feature 4: NumPy Activity Heatmap**

Creates a text based heatmap showing message activity by hour for each participant using NumPy.

In [ ]:
# Feature 4: NumPy Activity Heatmap
# AI-assisted: Asked AI how to build a heatmap (syntax for the heatmap)
def create_activity_heatmap(messages): # User defined function to create an activity heatmap
    participants = sorted(set(m['sender'] for m in messages)) # Get unique participants is sorted order
    hour_bins = [0, 3, 6, 9, 12, 15, 18, 21] # Define hour bins as 3-hour intervals for cleaner display
    hour_labels = ['00', '03', '06', '09', '12', '15', '18', '21']
    matrix = np.zeros((len(participants), len(hour_bins)), dtype=int) # Initialize the NumPy matrix
    for msg in messages:  # Count messages per person per hour bin
        sender = msg['sender']
        timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        hour = timestamp.hour
        bin_idx = 0 # Find which bin this hour belongs to
        for i, bin_start in enumerate(hour_bins):
            if hour >= bin_start:
                bin_idx = i
            else:
                break
        row_idx = participants.index(sender)
        matrix[row_idx, bin_idx] += 1
    return participants, hour_labels, matrix
participants, hour_labels, activity_matrix = create_activity_heatmap(messages) # Create the heatmap
print("  ACTIVITY HEATMAP (messages by hour)") # Print the formatted heatmap
print("           ", end="")
for label in hour_labels:
    print(f"  {label} ", end="")
print()
for i, person in enumerate(participants):
    print(f"  {person:<10}", end="")
    row_max = activity_matrix[i].max() if activity_matrix[i].max() > 0 else 1
    for j in range(len(hour_labels)):
        value = activity_matrix[i, j]
        if value == 0:
            char = '.'
        elif value < row_max * 0.25:
            char = '.'
        elif value < row_max * 0.5:
            char = '░'
        elif value < row_max * 0.75:
            char = '▒'
        else:
            char = '█'
        print(f"  {char}  ", end="")
    print()



  ACTIVITY HEATMAP (messages by hour)
             00   03   06   09   12   15   18   21 
  Aman        █    █    .    .    .    .    .    ░  
  Karan       .    .    .    ▒    █    █    █    ░  
  Neha        .    .    ░    █    ▒    ▒    █    ▒  
  Priya       .    .    ░    █    █    ▒    ▒    ░  
  Rahul       .    .    ░    ░    ▒    █    █    █  
  Vikas       .    .    ▒    ░    ▒    █    █    █  


## Feature 5: Top Words

Extracts and counts words from all messages, identifies the top 10 most-used words, and displays them with visual bars.

In [ ]:
# Feature 5: Word Frequency Analysis
def analyze_word_frequency(messages): # User defined function to analyze the word frequency
    stop_words = { # Define the stop words to skip
        'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
        'with', 'at', 'by', 'from', 'as', 'be', 'this', 'that', 'it', 'not',
        'are', 'was', 'were', 'have', 'has', 'had', 'do', 'does', 'did',
        'but', 'if', 'so', 'my', 'your', 'we', 'you', 'he', 'she', 'they',
        'me', 'him', 'her', 'them', 'us', 'what', 'when', 'where', 'who',
        'why', 'how', 'just', 'like', 'get', 'got', 'go', 'going', 'went'
    }
    punctuation = ".,!?;:'\"()[]{}<>-_/\\|@#$%^&*~`" # All the punctuation to strip
    word_counts = {}
    for msg in messages:
        text = msg['message'].lower()
        for char in punctuation: # Strip the punctuation
            text = text.replace(char, ' ')
        words = text.split() # Split into words
        for word in words: # Count the words (skip the stop words and short words)
            if len(word) < 2 or word in stop_words:
                continue
            word_counts[word] = word_counts.get(word, 0) + 1
    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]  # Get the top 10 words
    return sorted_words
top_words = analyze_word_frequency(messages) # Analyze the word frequency
print("  THIS GROUP'S FAVOURITE WORDS") # Print the formatted word frequency
max_count = top_words[0][1] if top_words else 1
for word, count in top_words:
    bar_length = int((count / max_count) * 30)
    bar = '█' * bar_length
    print(f"    {word:<12} {bar:<30} {count}")

  THIS GROUP'S FAVOURITE WORDS
    guys         ██████████████████████████████ 318
    today        ███████████████████████████    292
    about        █████████████████████████      274
    hai          █████████████████████████      268
    am           ████████████████████████       260
    his          ████████████████████           217
    everyone     ███████████████████            203
    which        ███████████████████            202
    telling      ████████████████               179
    up           ████████████████               172


## Feature 6: Response Speed & Silent Streaks

Computes average response time for each participant and identifies their longest silent streak (consecutive days with zero messages).

In [ ]:
# Feature 6: Response Speed & Silent Streaks
def compute_response_patterns(messages): # User defined function to compute the response paterns
    if not messages:
        return {}
    sorted_messages = sorted(messages, key=lambda x: datetime.strptime(x['timestamp'], '%d/%m/%y, %H:%M')) # Sort the messages by timestamp
    participants = set(m['sender'] for m in messages)  # Get the participants
    response_times = {p: [] for p in participants}  # Initialize the data structures
    active_days = {p: set() for p in participants}
    for i in range(1, len(sorted_messages)): # Compute the response times
        current_msg = sorted_messages[i]
        prev_msg = sorted_messages[i-1]
        if current_msg['sender'] != prev_msg['sender']:  # Only compute if the sender is different
            current_time = datetime.strptime(current_msg['timestamp'], '%d/%m/%y, %H:%M')
            prev_time = datetime.strptime(prev_msg['timestamp'], '%d/%m/%y, %H:%M')
            gap_seconds = (current_time - prev_time).total_seconds()
            response_times[current_msg['sender']].append(gap_seconds)
    for msg in sorted_messages: # Track the active days for each person
        timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        date = timestamp.date()
        active_days[msg['sender']].add(date)
    avg_response_times = {}  # Compute the average response times
    for person in participants:
        times = response_times[person]
        if times:
            avg_response_times[person] = sum(times) / len(times)
        else:
            avg_response_times[person] = float('inf')
    all_dates = sorted(set(m.date() for m in [datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M') for msg in messages]))  # Compute the longest silent streaks
    silent_streaks = {}
    for person in participants:
        person_active = active_days[person]
        max_streak = 0
        current_streak = 0
        streak_start = None
        streak_end = None
        for date in all_dates:
            if date not in person_active:
                current_streak += 1
                if streak_start is None:
                    streak_start = date
                streak_end = date
            else:
                if current_streak > max_streak:
                    max_streak = current_streak
                current_streak = 0
                streak_start = None
        if current_streak > max_streak:  # Check the final streak
            max_streak = current_streak
        silent_streaks[person] = {
            'days': max_streak,
            'start': streak_start if max_streak > 0 else None,
            'end': streak_end if max_streak > 0 else None
        }
    return {
        'avg_response_times': avg_response_times,
        'silent_streaks': silent_streaks
    }
response_data = compute_response_patterns(messages) # Compute the response patterns
avg_times = response_data['avg_response_times'] # Find the fastest and slowest repliers
fastest = min(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else float('inf'))
slowest = max(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else 0)
print("  RESPONSE PATTERNS") # Print the response patterns
if fastest[1] != float('inf'):
    fastest_minutes = fastest[1] / 60
    print(f"    Fastest replier : {fastest[0]} (avg {fastest_minutes:.1f} minutes)")
else:
    print(f"    Fastest replier : N/A")
if slowest[1] != float('inf'):
    if slowest[1] >= 3600:
        slowest_hours = slowest[1] / 3600
        print(f"    Slowest replier : {slowest[0]} (avg {slowest_hours:.1f} hours)")
    else:
        slowest_minutes = slowest[1] / 60
        print(f"    Slowest replier : {slowest[0]} (avg {slowest_minutes:.1f} minutes)")
else:
    print(f"    Slowest replier : N/A")
print()
print("  LONGEST SILENT STREAKS (consecutive days with zero messages)")
silent_streaks = response_data['silent_streaks'] # Sort by the silent streak length
sorted_streaks = sorted(silent_streaks.items(), key=lambda x: x[1]['days'], reverse=True)
for person, streak in sorted_streaks:
    if streak['days'] == 0:
        print(f"    {person:<10} :  0 days  (never went silent)")
    elif streak['days'] == 1:
        print(f"    {person:<10} :  {streak['days']} day")
    else:
        if streak['start'] and streak['end']:
            start_str = streak['start'].strftime('%d %b')
            end_str = streak['end'].strftime('%d %b')
            print(f"    {person:<10} : {streak['days']:>2} days ({start_str} to {end_str})")
        else:
            print(f"    {person:<10} : {streak['days']:>2} days")

  RESPONSE PATTERNS
    Fastest replier : Vikas (avg 34.9 minutes)
    Slowest replier : Aman (avg 54.9 minutes)

  LONGEST SILENT STREAKS (consecutive days with zero messages)
    Vikas      : 11 days (28 May to 30 May)
    Neha       :  0 days  (never went silent)
    Karan      :  0 days  (never went silent)
    Priya      :  0 days  (never went silent)
    Aman       :  0 days  (never went silent)
    Rahul      :  0 days  (never went silent)


## Feature 7: Personality Archetype Detection

Assigns personality archetypes to each participant based on quantitative rules including: Spammer, Group Mom, Night Owl, Storyteller, Drama Queen, Ghost, Comedian, and Question Master.

In [ ]:
# Feature 7: Personality Archetype Detection
# AI-assisted: Asked AI how to detect the archetype for each participant
def detect_archetypes(messages):
    participants = set(m['sender'] for m in messages)
    sorted_messages = sorted(messages, key=lambda x: datetime.strptime(x['timestamp'], '%d/%m/%y, %H:%M'))  # Sort the messages by timestamp for analysis
    archetype_scores = {p: {} for p in participants}  # Initialize the scores for each archetype
    for person in participants:  # THE SPAMMER: Avg consecutive message burst > 3
        person_messages = [m for m in sorted_messages if m['sender'] == person]
        bursts = []
        current_burst = 1
        for i in range(1, len(person_messages)):
            current_idx = sorted_messages.index(person_messages[i]) # Find the position in overall message list
            prev_idx = sorted_messages.index(person_messages[i-1])
            if current_idx == prev_idx + 1:  # If consecutive in overall list (no one else spoke)
                current_burst += 1
            else:
                bursts.append(current_burst)
                current_burst = 1
        bursts.append(current_burst)
        avg_burst = sum(bursts) / len(bursts) if bursts else 0
        archetype_scores[person]['SPAMMER'] = avg_burst
    caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',  # THE GROUP MOM: Caring keywords
                      'reminder', 'drink water', 'don\'t forget', 'careful', 'well']
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        caring_count = 0
        for msg in person_messages:
            text_lower = msg['message'].lower()
            for keyword in caring_keywords:
                if keyword in text_lower:
                    caring_count += 1
                    break
        percentage = (caring_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['GROUP_MOM'] = percentage
    for person in participants:  # THE NIGHT OWL: Messages between 23:00 and 04:59
        person_messages = [m for m in messages if m['sender'] == person]
        night_count = 0
        for msg in person_messages:
            timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
            hour = timestamp.hour
            if hour >= 23 or hour < 5:
                night_count += 1
        percentage = (night_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['NIGHT_OWL'] = percentage
    for person in participants:  # THE STORYTELLER: Average words per message > 30
        person_messages = [m for m in messages if m['sender'] == person]
        total_words = 0
        for msg in person_messages:
            words = msg['message'].split()
            total_words += len(words)
        avg_words = total_words / len(person_messages) if person_messages else 0
        archetype_scores[person]['STORYTELLER'] = avg_words
    for person in participants:  # THE DRAMA QUEEN: ALL-CAPS or multiple exclamations
        person_messages = [m for m in messages if m['sender'] == person]
        drama_count = 0
        for msg in person_messages:
            text = msg['message']
            if len(text) > 3 and text.isupper() and text.isalpha(): # Check if all the caps and length > 3
                drama_count += 1
            elif text.count('!') >= 2: # Check for 2+ exclamations
                drama_count += 1
        percentage = (drama_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['DRAMA_QUEEN'] = percentage
    all_dates = sorted(set(datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M').date() for m in messages))  # THE GHOST: Silent on > 60% of days
    total_days = len(all_dates)
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        active_dates = set(datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M').date() for m in person_messages)
        silent_days = total_days - len(active_dates)
        silent_percentage = (silent_days / total_days) * 100 if total_days > 0 else 0
        archetype_scores[person]['GHOST'] = silent_percentage
    laughter_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']  # THE COMEDIAN: Laughter words
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        laughter_count = 0
        for msg in person_messages:
            text_lower = msg['message'].lower()
            for word in laughter_words:
                if word in text_lower:
                    laughter_count += 1
                    break
        percentage = (laughter_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['COMEDIAN'] = percentage
    for person in participants:  # THE QUESTION MASTER: Messages ending with '?'
        person_messages = [m for m in messages if m['sender'] == person]
        question_count = 0
        for msg in person_messages:
            if msg['message'].strip().endswith('?'):
                question_count += 1
        percentage = (question_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['QUESTION_MASTER'] = percentage
    archetype_names = {  # Assign best archetype to each person
        'SPAMMER': 'THE SPAMMER',
        'GROUP_MOM': 'THE GROUP MOM',
        'NIGHT_OWL': 'THE NIGHT OWL',
        'STORYTELLER': 'THE STORYTELLER',
        'DRAMA_QUEEN': 'THE DRAMA QUEEN',
        'GHOST': 'THE GHOST',
        'COMEDIAN': 'THE COMEDIAN',
        'QUESTION_MASTER': 'THE QUESTION MASTER'
    }
    assignments = {}
    for person in participants:
        scores = archetype_scores[person]
        best_archetype = max(scores.items(), key=lambda x: x[1])
        assignments[person] = {
            'archetype': archetype_names[best_archetype[0]],
            'score': best_archetype[1]
        }
    return assignments, archetype_scores
archetype_assignments, all_scores = detect_archetypes(messages) # Detect the archetypes
print("  PERSONALITY ARCHETYPES") # Print the archetype assignments
for person in sorted(archetype_assignments.keys()):
    assignment = archetype_assignments[person]
    archetype = assignment['archetype']
    score = assignment['score']
    if 'SPAMMER' in archetype: # Format the score based on the archetype
        print(f"    {person:<10} →  {archetype:<20} (avg {score:.1f} msgs in a row)")
    elif 'GROUP MOM' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% caring keywords)")
    elif 'NIGHT OWL' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% msgs after 11 PM)")
    elif 'STORYTELLER' in archetype:
        print(f"    {person:<10} →  {archetype:<20} (avg {score:.1f} words per msg)")
    elif 'DRAMA QUEEN' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% ALL-CAPS msgs)")
    elif 'GHOST' in archetype:
        print(f"    {person:<10} →  {archetype:<20} (silent {score:.1f}% of days)")
    elif 'COMEDIAN' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% laughter)")
    elif 'QUESTION MASTER' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% questions)")

  PERSONALITY ARCHETYPES
    Aman       →  THE NIGHT OWL        (80.4% msgs after 11 PM)
    Karan      →  THE STORYTELLER      (avg 57.0 words per msg)
    Neha       →  THE DRAMA QUEEN      (8.8% ALL-CAPS msgs)
    Priya      →  THE GROUP MOM        (67.0% caring keywords)
    Rahul      →  THE NIGHT OWL        (13.5% msgs after 11 PM)
    Vikas      →  THE GHOST            (silent 73.3% of days)


## Feature 8: Final Report

Generates a comprehensive, beautifully formatted report combining all features with proper visual hierarchy and formatting.

In [ ]:
# Feature 8: Final Report

def generate_final_report(messages, overview, top_words, participants, hour_labels, activity_matrix,
                          response_data, archetype_assignments, activity_periods): # User defined function to generate the final report
    print("=" * 70)
    print(f'GROUPDNA REPORT - "{overview["group_name"]}"')
    print(f"{overview['total_days']} days • {overview['total_messages']:,} messages • {overview['participants']} members")
    print("=" * 70)
    print()
    print(f"Period         : {overview['first_date']} to {overview['last_date']}")  # Period and Busiest Times Section
    print(f"Busiest day    : {activity_periods['most_active_day']['date']} ({activity_periods['most_active_day']['count']} messages)")
    hour = activity_periods['most_active_hour']['hour']
    next_hour = (hour + 1) % 24
    avg = activity_periods['most_active_hour']['avg_per_day']
    print(f"Busiest hour   : {hour:02d}:00 - {next_hour:02d}:00 (avg {avg:.1f} messages per day)")
    print()
    print("MESSAGES PER PERSON")  # Messages Per Person Section
    total_msgs = overview['total_messages']
    max_bar_length = 20
    for sender, count in overview['message_counts']:
        percentage = (count / total_msgs) * 100
        bar_length = int((percentage / 100) * max_bar_length)
        bar = '█' * bar_length
        print(f"{sender:<10} {bar:<{max_bar_length}} {count:>5} ({percentage:>5.1f}%)")
    print()
    print("ACTIVITY HEATMAP")  # Activity Heatmap Section
    print("           ", end="")
    for label in hour_labels:
        print(f"  {label} ", end="")
    print()
    for i, person in enumerate(participants):
        print(f"  {person:<10}", end="")
        row_max = activity_matrix[i].max() if activity_matrix[i].max() > 0 else 1
        for j in range(len(hour_labels)):
            value = activity_matrix[i, j]
            if value == 0:
                char = '.'
            elif value < row_max * 0.25:
                char = '.'
            elif value < row_max * 0.5:
                char = '░'
            elif value < row_max * 0.75:
                char = '▒'
            else:
                char = '█'
            print(f"  {char}  ", end="")
        assignment = archetype_assignments[person]  # Add the archetype annotation for Night Owl
        if 'NIGHT OWL' in assignment['archetype']:
            print(" <- NIGHT OWL", end="")
        print()
    print()
    print("THIS GROUP'S FAVOURITE WORDS")  # Top Words Section (Top 5 instead of Top 10)
    top_5_words = top_words[:5]
    max_count = top_5_words[0][1] if top_5_words else 1
    for word, count in top_5_words:
        bar_length = int((count / max_count) * 20)
        bar = '█' * bar_length
        print(f"{word:<12} {bar:<20} {count}")
    print()
    print("RESPONSE PATTERNS")  # Response Patterns Section
    avg_times = response_data['avg_response_times']
    fastest = min(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else float('inf'))
    slowest = max(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else 0)
    if fastest[1] != float('inf'):
        fastest_minutes = fastest[1] / 60
        print(f"Fastest replier : {fastest[0]} (avg {fastest_minutes:.1f} minutes)")
    else:
        print(f"Fastest replier : N/A")
    if slowest[1] != float('inf'):
        if slowest[1] >= 3600:
            slowest_hours = slowest[1] / 3600
            print(f"Slowest replier : {slowest[0]} (avg {slowest_hours:.1f} hours)")
        else:
            slowest_minutes = slowest[1] / 60
            print(f"Slowest replier : {slowest[0]} (avg {slowest_minutes:.1f} minutes)")
    else:
        print(f"Slowest replier : N/A")
    print()
    print("LONGEST SILENT STREAKS")  # Longest Silent Streaks Section
    silent_streaks = response_data['silent_streaks']
    sorted_streaks = sorted(silent_streaks.items(), key=lambda x: x[1]['days'], reverse=True)
    for person, streak in sorted_streaks:
        if streak['days'] == 0:
            continue  # Skip people who never went silent
        elif streak['days'] == 1:
            print(f"{person:<10} : {streak['days']} day")
        else:
            if streak['start'] and streak['end']:
                start_str = streak['start'].strftime('%d %b')
                end_str = streak['end'].strftime('%d %b')
                print(f"{person:<10} : {streak['days']} days ({start_str} - {end_str})")
            else:
                print(f"{person:<10} : {streak['days']} days")
    print()
generate_final_report(messages, overview, top_words, participants, hour_labels,
                      activity_matrix, response_data, archetype_assignments, activity_periods) # Generate the final report

GROUPDNA REPORT - "Hostel Bois 4ever"
60 days • 3,127 messages • 6 members

Period         : 01 April 2024 to 30 May 2024
Busiest day    : 04 May 2024 (244 messages)
Busiest hour   : 18:00 - 19:00 (avg 4.1 messages per day)

MESSAGES PER PERSON
Rahul      ██████                 940 ( 30.1%)
Priya      ████                   712 ( 22.8%)
Neha       ███                    624 ( 20.0%)
Aman       ███                    484 ( 15.5%)
Karan      ██                     345 ( 11.0%)
Vikas                              22 (  0.7%)

ACTIVITY HEATMAP
             00   03   06   09   12   15   18   21 
  Aman        █    █    .    .    .    .    .    ░   <- NIGHT OWL
  Karan       .    .    .    ▒    █    █    █    ░  
  Neha        .    .    ░    █    ▒    ▒    █    ▒  
  Priya       .    .    ░    █    █    ▒    ▒    ░  
  Rahul       .    .    ░    ░    ▒    █    █    █   <- NIGHT OWL
  Vikas       .    .    ▒    ░    ▒    █    █    █  

THIS GROUP'S FAVOURITE WORDS
guys         ████████████████

---

**Reflection**

**What was the hardest part?**
The most challenging part of this project was implementing the archetype detection logic and particularly the consecutive message burst calculation for the Spammer archetype. Getting the burst detection to correctly identify when someone sends multiple messages in a row without others speaking required careful index tracking and edge case handling.

**What would you do differently?**
If I were to redo this project then I would:
1. Start with unit tests for each feature to catch bugs earlier
2. Create helper functions for common operations like datetime parsing to reduce code duplication
3. Add more robust error handling for edge cases in the chat format
4. Implement the multi-line message handling from the start (even though the dataset does not need it)

**What archetype did I get?**

[Optional: Run this on your own chat and share your archetype here!]

---

**Project Constraints Followed**

-  No pandas used for file reading (used Python's built-in `open()`)
-  No matplotlib used for heatmap (used text-based rendering with NumPy)
-  No collections.Counter used (built custom word counter with dict)
-  Only Python fundamentals + NumPy + datetime used
-  All 8 features implemented
-  Proper file I/O with UTF-8 encoding
-  Clean, formatted output with visual hierarchy
-  Proper error handling for edge cases

---

**Note:** This notebook can be run end-to-end by uploading `hostel_bois.txt` to the same directory in Google Colab or placing it next to this .ipynb file locally.